# Please Download Real Estate csv to run project 
url = https://drive.google.com/file/d/1fFFzQI9X2GKbkeTWt04QzXtZzKhww1Ek/view?usp=sharing

In [ ]:
import pandas as pd 

In [ ]:
real_estate = pd.read_csv ("egypt_real_estate_listings.csv")
real_estate

In [ ]:
locations_part = real_estate["location"].str.split (",")
locations_part

In [ ]:
real_estate ["Goveroment"] = locations_part.apply (lambda x :  x[-1] if type (x) == list and len (x) >0 else None )
real_estate["City"] = locations_part.apply (lambda x : x[-2] if type (x) == list  and len(x) > 2  else  ( x[0] if type(x) == list else None ))
real_estate 

In [ ]:
real_estate = real_estate.drop (columns = ["url"])
real_estate = real_estate.dropna (subset = ["down_payment"])
real_estate

In [ ]:
def get_context_for_LLM ( real_estate_df , gov , city  , top_k = 10  ) :
    gov_filtered = real_estate_df["Goveroment"].str.contains(gov.strip() , case = False , na = False  )
    city_filtered = real_estate_df["City"].str.contains(city.strip () , case = False , na = False )
    filtered_df = real_estate[gov_filtered & city_filtered]

    result = filtered_df.head(top_k)
    if result.empty : 
        return "Sorry I don't found any real estate with this search "
    
    context_lines = [ ]
    for _, row in result.iterrows ( ) :
        line = f"Real esatae  in  {row["location"]}  , Price : {row["price"]} ,  type  : {row["type"]} , number of bedrooms :  {row["bedrooms"]} , number of bathroom  : {row["bathrooms"]} size : {row["size"]}"
        context_lines.append(line)
    return "\n\n".join(context_lines)

In [ ]:
import os 
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
load_dotenv(override=True)
Model = "gpt-5-mini"
openai = OpenAI()

In [ ]:
def Real_estate_Assistant (user_prompt  , gov , city , top_k  ) : 


    system_prompt = """
        you are a highly skilled real estate wholesaler , Professional in helping buyers find the right property real_estate,
        your role and Objectives : 
        identify and qualify serious cash buyers 
        specify the type of property required , the number of rooms and bathrooms and the budget 
        your style and communication Guidelines : 
        Be as concise as possible; don't talk too much.
        Act likes a seasoned real estate wholesaler with years of experience .

        These are some of the results based on the governorate and city the user requests : 
         """
    system_prompt += get_context_for_LLM ( real_estate , gov , city  , top_k  )
    
    Mesage = [
        {"role" : "system" , "content" : system_prompt } , 
        {"role" : "user" , "content" : user_prompt }
    ]

    stream = openai.chat.completions.create( 
        model = Model , 
        messages = Mesage ,
        stream = True 
        ) 
    result = ""
    for chuck in stream : 
        result += chuck.choices[0].delta.content or ""
        yield result 

In [ ]:
import gradio as gr 

In [ ]:
def City_Choice (gov_name) : 
    if not gov_name : 
        return gr.Dropdown (choices = [] , value = None)
    clean_gov = str ( gov_name ).lower ().strip()
    filtered_df = real_estate[ real_estate["Goveroment"].str.lower().str.strip ( ) == clean_gov]
    cities_list = filtered_df['City'].dropna().unique().tolist()
    return gr.Dropdown(choices=cities_list, value=cities_list[0] if cities_list else None)

In [ ]:
gov_choice = real_estate["Goveroment"].unique().tolist()


In [ ]:
import gradio as gr

In [ ]:
with gr.Blocks() as view : 
    gr.Markdown ( "# Real Estate AI  Assistant  ")

    with gr.Row () : 
        
        select_GOV = gr.Dropdown (
        choices =gov_choice  , 
        label = "Select GOV " , 
        value = "Cairo" )

        select_City = gr.Dropdown ( 
            choices = [] , 
            label = "Selected City " , 
            value  = None ,
        )
        num_of_real_estate = gr.Dropdown ( choices = list (range (1 , 20) )  ,  label = "Specify the number of properties you want to display." , value = 5)

    message_input = gr.Textbox (label = "Enter your message " , info = "Enter a message for the LLM " , lines = 3 )
    message_output = gr.Markdown (label = "Response" )
    submit_btn = gr.Button ("Send")

    submit_btn.click ( 
        fn = Real_estate_Assistant ,
        inputs = [message_input , select_GOV , select_City , num_of_real_estate ] ,
        outputs = [message_output],
  
    )

    select_GOV.change(
        fn= City_Choice , 
        inputs = select_GOV,
        outputs = select_City
    )


    view.launch(inbrowser=True)